# Module 13: Agent Harness & Loop Engineering

**Goal:** Build agents that don't get stuck, crash, or over-run.

**What you'll learn:**
- Loop-until-dry with novelty gates
- Budget-aware loops with token tracking
- Durable journals for crash-proof resume
- Self-repair loops with retry logic

**Prerequisites:** Module 07 (agentic workflows basics)

---

## 0. Setup

In [ ]:
import hashlib
import json
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass

print("✅ Setup complete")

---
## 1. Novelty Gate — Loop-Until-Dry

Keep running until K consecutive rounds produce nothing new.

In [ ]:
class NoveltyGate:
    def __init__(self, dry_threshold: int = 3):
        self.seen = set()
        self.dry_rounds = 0
        self.dry_threshold = dry_threshold

    def fingerprint(self, item):
        return hashlib.md5(str(item).encode()).hexdigest()

    def filter_new(self, items):
        new_items = [i for i in items if self.fingerprint(i) not in self.seen]
        if new_items:
            for item in new_items:
                self.seen.add(self.fingerprint(item))
            self.dry_rounds = 0
        else:
            self.dry_rounds += 1
        return new_items

    @property
    def is_dry(self):
        return self.dry_rounds >= self.dry_threshold

# Simulate: 5 new items → 3 new → 0 → 0 → 0 (should stop)
gate = NoveltyGate(dry_threshold=3)
rounds = [[1, 2, 3, 4, 5], [6, 7, 8], [], [], []]

for i, items in enumerate(rounds):
    fresh = gate.filter_new(items)
    print(f"Round {i+1}: {len(items)} items → {len(fresh)} new, dry={gate.dry_rounds}, is_dry={gate.is_dry}")
    if gate.is_dry:
        print(f"  → Stopped after round {i+1}")
        break

---
## 2. Budget-Aware Loop

Stop when token budget is exhausted. Inject remaining budget into agent context.

In [ ]:
@dataclass
class BudgetTracker:
    total_tokens: int
    spent_tokens: int = 0

    def record(self, input_tokens: int, output_tokens: int):
        self.spent_tokens += input_tokens + output_tokens

    @property
    def remaining(self):
        return max(0, self.total_tokens - self.spent_tokens)

    @property
    def is_exhausted(self):
        return self.remaining < 1000

# Simulate a loop consuming budget
budget = BudgetTracker(total_tokens=10_000)
iteration = 0

while not budget.is_exhausted:
    iteration += 1
    # Simulate: each iteration uses ~2000 tokens
    budget.record(input_tokens=1500, output_tokens=500)
    print(f"Iteration {iteration}: spent={budget.spent_tokens:,}, remaining={budget.remaining:,}")

print(f"\nStopped after {iteration} iterations (budget nearly exhausted)")

---
## 3. Durable Journal — Crash-Proof Resume

Write each completed step to an append-only log. On restart, replay to skip completed work.

In [ ]:
class Journal:
    def __init__(self, run_id: str, journal_dir: str = ".journals"):
        self.path = Path(journal_dir) / f"{run_id}.jsonl"
        self.path.parent.mkdir(exist_ok=True)
        self._completed = {}
        self._load()

    def _load(self):
        if self.path.exists():
            with open(self.path) as f:
                for line in f:
                    entry = json.loads(line)
                    self._completed[entry["step_id"]] = entry["result"]
            print(f"  Resumed: {len(self._completed)} steps already done")

    def step_id(self, name, *args):
        return hashlib.md5(f"{name}:{':'.join(str(a) for a in args)}".encode()).hexdigest()[:12]

    def is_done(self, sid):
        return sid in self._completed

    def execute(self, name, fn, *args):
        sid = self.step_id(name, *args)
        if self.is_done(sid):
            print(f"  ↩ Skipping '{name}' (already done)")
            return self._completed[sid]
        result = fn(*args)
        with open(self.path, "a") as f:
            f.write(json.dumps({"step_id": sid, "step_name": name, "result": result, "timestamp": datetime.now().isoformat()}) + "\n")
        self._completed[sid] = result
        print(f"  ✓ Completed '{name}'")
        return result

# Demo: run, crash, resume
journal = Journal(run_id="demo_run")
results = []
for topic in ["context engineering", "agent harness", "MCP"]:
    result = journal.execute("summarize", lambda t: f"Summary of {t}", topic)
    results.append(result)

print(f"\nResults: {results}")

---
## Summary

| Pattern | Purpose |
|---------|---------|
| **Novelty Gate** | Stop when no new items found (exhaustive search) |
| **Budget Tracker** | Stop when token/cost budget exhausted |
| **Durable Journal** | Crash-proof resume without re-executing completed steps |
| **Self-Repair** | Retry with error context fed back to agent |

**The harness is what separates a demo from a production agent. Build it first.**

---
## 🧪 Exercises

1. **Novelty Gate Calibration**: Simulate rounds producing 5, 3, 0, 0, 0 items. With threshold=3, does it stop after round 3 or round 5? Why?

2. **Journal Resume**: Write a 5-step pipeline. After step 3, raise an exception. On restart, verify steps 1-3 are skipped.

3. **Budget Governor**: Run a loop with 50,000 token budget, ~5,000 tokens/iteration. Verify it stops before exceeding budget.